In [ ]:
import  scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df_types = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeTypeText.pkl")
df_types.index = df_types.Date#["2020-05-31":]
df_dates = df_types["2020-05-31":"2022-06-01"]
df_dates = df_dates.loc[df_dates["Type"]=="DIVIDEND"]

tickers = df_dates.Company.values[1:5]
tickers = list(dict.fromkeys(tickers))
df_dates = df_dates.loc[df_dates['Company'].isin(tickers)]

In [ ]:
len(tickers)

In [ ]:
events_db = []
for date, row in df_dates.T.iteritems():
    print(date)
    new_row = {'event_date': np.datetime64(row["Date"]),'market_ticker':'^VIX','security_ticker': row["Company"]}
    events_db.append(new_row)

In [ ]:
len(df_dates)

In [ ]:
type(events_db)

In [ ]:
len(events_db)

In [ ]:
from multiprocessing.resource_sharer import stop
from yahoo_fin.stock_info import get_data
# del returns
returns = pd.DataFrame()

######################## VIX #########################
stock_name = get_data("^VIX", start_date="2020-05-01", end_date="2022-08-01", index_as_date = True, interval="1d")
stock_name.index.rename("date",inplace=True)
# stock_name = web.get_data_yahoo(ticker,
#                             start = "2020-06-01",
#                             end = "2022-06-01")
stock_name_daily_returns = stock_name['adjclose'].pct_change()
df = stock_name_daily_returns.to_frame()
df.index.rename("date",inplace=True)
df.rename(columns={"adjclose":"^VIX"},inplace=True)
returns = df.join(returns)
######################## SPY #########################
# stock_name = get_data("SPY", start_date="2020-05-01", end_date="2022-08-01", index_as_date = True, interval="1d")
# stock_name.index.rename("date",inplace=True)
# # stock_name = web.get_data_yahoo(ticker,
# #                             start = "2020-06-01",
# #                             end = "2022-06-01")
# stock_name_daily_returns = stock_name['adjclose'].pct_change()
# df = stock_name_daily_returns.to_frame()
# df.index.rename("date",inplace=True)
# df.rename(columns={"adjclose":"SPY"},inplace=True)
# returns = pd.merge(df, returns, on='date')
################## S&P500 tickers ####################
for ticker in tickers:
    try:
        print(ticker)
        stock_name = get_data(ticker, start_date="2020-05-01", end_date="2022-08-01", index_as_date = True, interval="1d")
        stock_name.index.rename("date",inplace=True)
        stock_name_daily_returns = stock_name['adjclose'].pct_change()
        df = stock_name_daily_returns.to_frame()
        df.index.rename("date",inplace=True)
        df.rename(columns={"adjclose":str(ticker)},inplace=True)
        returns = pd.merge(df, returns, on='date')
        # print(df)#={"":str(ticker)})
    except:
        print("EXCEPTION"+ticker)
returns.drop(index=returns.index[0], 
             axis=0, 
             inplace=True)

In [ ]:
returns

In [ ]:
returns.drop(index=returns.index[0], 
             axis=0, 
             inplace=True)

In [ ]:
returns.tail(150)

In [ ]:
returns

In [ ]:
returns.to_csv("returns_list.csv")

In [ ]:
es.Single.import_returns("returns_list.csv")
# es.Single.import_FamaFrench("FamaFrenchSPY.csv")

In [ ]:
# tickers = ['SPY']
releases = dict()
for ticker in tickers:
    try:
        events = [event for event in events_db if event['security_ticker']==ticker]
        print(ticker)
        releases[ticker] = es.Multiple.from_list(
            events,
            es.Single.market_model,#FamaFrench_3factor,
            event_window= (-5,+30)
        )
        print("##########################")
    except:
        print("EXCEPTION: "+ticker)

In [ ]:
len(events)

In [ ]:
releases

In [ ]:
ticker

In [ ]:
for ticker, event in releases.items():
    print(ticker)
    event.plot()
    plt.show()